# Hands-On Minimal Toy Tutorial: Build the Entire Pipeline from Scratch!
### Quantitative StatArb Practice Lab (Just 3 Coins & 5 Timestamps)

Welcome! This workbook is designed to give you complete, 100% crystal-clear transparency into the entire quantitative trading pipeline.

Instead of analyzing thousands of bars across 10 coins, we use a **tiny toy universe**:
* **3 Coins:** `BTC`, `ETH`, `SOL`
* **5 Timestamps:** `T1`, `T2`, `T3`, `T4`, `T5`

Every number here is so simple you could calculate it with a pen, paper, and a handheld calculator in 2 minutes!

---

## The 5-Step Pipeline You Will Build:
1. **Step 1:** Load the raw price table (`toy_crypto_prices.csv`).
2. **Step 2:** Calculate asset percentage returns.
3. **Step 3:** Generate the momentum signal and calculate **Dollar-Neutral Weights** (Rank $\rightarrow$ Demean $\rightarrow$ Normalize).
4. **Step 4:** Calculate **Strategy Returns** (using `weights.shift(1)` to prevent lookahead bias).
5. **Step 5:** Calculate the **Sharpe Ratio**!


---
## Step 1: Load the Raw Price Table

Here are the starting prices:
* **T1:** `BTC = 100`, `ETH = 50`, `SOL = 10`
* **T2:** `BTC = 110` (+10%), `ETH = 50` (0%), `SOL = 9` (-10%)
* ...and so on.

Let's load this table into Pandas and inspect it.


In [1]:
import os
import pandas as pd
import numpy as np

# Load the toy dataset
data_path = os.path.join(os.getcwd(), 'data', 'toy_crypto_prices.csv')
prices = pd.read_csv(data_path, index_col=0)

print("=== RAW PRICE TABLE ===")
display(prices)


=== RAW PRICE TABLE ===


,BTC,ETH,SOL
T1,100.00,50.000,10.000
T2,110.00,50.000,9.000
T3,104.50,52.500,9.900
T4,114.95,52.500,8.910
T5,103.45,55.125,9.801


---
## Step 2: Calculate Asset Percentage Returns

### The Math:
$$\text{Return}_t = \frac{\text{Price}_t - \text{Price}_{t-1}}{\text{Price}_{t-1}}$$

### Pen & Paper Check (from T1 to T2):
* **BTC:** $\frac{110 - 100}{100} = +0.10$ (+10%)
* **ETH:** $\frac{50 - 50}{50} = 0.00$ (0%)
* **SOL:** $\frac{9 - 10}{10} = -0.10$ (-10%)

Let's compute this in Python!


In [2]:
# Calculate returns using price difference divided by previous price
previous_prices = prices.shift(1)
price_changes = prices - previous_prices
returns = price_changes / previous_prices

# Notice T1 has no previous price, so we drop it
returns = returns.dropna()

print("=== ASSET RETURNS TABLE ===")
display(returns.round(3))


=== ASSET RETURNS TABLE ===


,BTC,ETH,SOL
T2,0.10,0.00,-0.1
T3,-0.05,0.05,0.1
T4,0.10,0.00,-0.1
T5,-0.10,0.05,0.1


---
## Step 3: Turn Signals into Dollar-Neutral Portfolio Weights

Now, we build a **1-period Momentum strategy**:
* If a coin was the top winner $\rightarrow$ We want to **BUY** it (positive weight).
* If a coin was the bottom loser $\rightarrow$ We want to **SHORT** it (negative weight).
* Total Longs must equal Total Shorts (Dollar-Neutral: sum = 0).
* Total absolute exposure must equal 1.0 (Unit Leverage: sum of abs = 1.0).

### Pen & Paper Check at T2:
At T2, the returns were: `SOL = -0.10`, `ETH = 0.00`, `BTC = +0.10`.
1. **Rank from lowest to highest:**
   * `SOL` is rank 1 (lowest)
   * `ETH` is rank 2 (middle)
   * `BTC` is rank 3 (highest)
2. **Calculate Average Rank:** $\frac{1 + 2 + 3}{3} = 2.0$
3. **Demean (Subtract 2.0 to center around zero):**
   * `SOL`: $1 - 2 = \mathbf{-1.0}$
   * `ETH`: $2 - 2 = \mathbf{0.0}$
   * `BTC`: $3 - 2 = \mathbf{+1.0}$
   * *Notice:* $-1.0 + 0.0 + 1.0 = \mathbf{0.0}$ (Dollar-neutral!).
4. **Normalize (Divide by sum of absolute values):**
   * Sum of absolute values $= |-1| + |0| + |+1| = 2.0$.
   * `SOL` weight: $-1.0 / 2.0 = \mathbf{-0.50}$ (Short \$0.50 of SOL)
   * `ETH` weight: $0.0 / 2.0 = \mathbf{0.00}$
   * `BTC` weight: $+1.0 / 2.0 = \mathbf{+0.50}$ (Long \$0.50 of BTC)
   * *Notice:* $|-0.5| + |0| + |+0.5| = \mathbf{1.0}$ (Unit leverage!).

Let's compute this for all rows in Python:


In [3]:
# Create an empty table for weights
weights = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)

for timestamp in returns.index:
    # 1. Get current returns at this timestamp (our signal)
    current_signal = returns.loc[timestamp]
    
    # 2. Rank coins from lowest (1) to highest (3)
    ranks = current_signal.rank()
    
    # 3. Demean (subtract average rank) -> forces sum to 0.0
    centered_ranks = ranks - ranks.mean()
    
    # 4. Normalize -> forces sum of absolute weights to 1.0
    total_abs = centered_ranks.abs().sum()
    if total_abs > 0:
        weights.loc[timestamp] = centered_ranks / total_abs

print("=== PORTFOLIO WEIGHTS TABLE ===")
display(weights.round(2))

# Verify rules:
print("\nVerification Check:")
print("1. Do weights sum to 0.0 (Dollar-Neutral)?\n", weights.sum(axis=1).round(6))
print("2. Do absolute weights sum to 1.0 (Unit Leverage)?\n", weights.abs().sum(axis=1).round(6))


=== PORTFOLIO WEIGHTS TABLE ===


,BTC,ETH,SOL
T2,0.5,0.0,-0.5
T3,-0.5,0.0,0.5
T4,0.5,0.0,-0.5
T5,-0.5,0.0,0.5



Verification Check:
1. Do weights sum to 0.0 (Dollar-Neutral)?
 T2    0.0
T3    0.0
T4    0.0
T5    0.0
dtype: float64
2. Do absolute weights sum to 1.0 (Unit Leverage)?
 T2    1.0
T3    1.0
T4    1.0
T5    1.0
dtype: float64


---
## Step 4: Calculate Strategy Returns (P&L)

Now, how much money did our strategy make?

### The Rule: Prevent Lookahead Bias!
* At timestamp **T2**, we decided our weights: Long BTC (+0.50), Short SOL (-0.50).
* We hold these positions from T2 to T3.
* Therefore, the return we earn at **T3** is:
  $$\text{Strategy Return at T3} = (\text{Weight from T2}) \times (\text{Asset Return at T3})$$
* That is why we use `weights.shift(1)`!

### Pen & Paper Check at T3:
* **Position decided at T2:** `BTC = +0.50`, `ETH = 0.00`, `SOL = -0.50`.
* **Asset returns that happened between T2 and T3:**
  * `BTC`: went down $-5.0\%$ ($-0.05$)
  * `ETH`: went up $+5.0\%$ ($+0.05$)
  * `SOL`: went up $+10.0\%$ ($+0.10$)
* **Our Portfolio Return at T3:**
  $$\text{Return} = (+0.50 \times -0.05) + (0.00 \times 0.05) + (-0.50 \times 0.10)$$
  $$\text{Return} = -0.025 + 0.00 - 0.050 = \mathbf{-0.075} \text{ (or } -7.5\%)$$
* *Notice:* Our momentum bet lost $-7.5\%$ because BTC reversed down and SOL reversed up!

Let's compute this in Python across all timestamps:


In [4]:
# Shift weights by 1 timestamp so yesterday's decision earns today's return
previous_weights = weights.shift(1)

# Multiply weights by asset returns
pnl_contributions = previous_weights * returns

# Sum across all coins to get the portfolio return at each timestamp
strategy_returns = pnl_contributions.sum(axis=1).dropna()

print("=== PnL CONTRIBUTIONS PER ASSET ===")
display(pnl_contributions.round(4))

print("=== TOTAL STRATEGY RETURN PER TIMESTAMP ===")
display(strategy_returns.round(4))


=== PnL CONTRIBUTIONS PER ASSET ===


,BTC,ETH,SOL
T2,NaN,NaN,NaN
T3,-0.025,0.0,-0.05
T4,-0.050,0.0,-0.05
T5,-0.050,0.0,-0.05


=== TOTAL STRATEGY RETURN PER TIMESTAMP ===


T2    0.000
T3   -0.075
T4   -0.100
T5   -0.100
dtype: float64

---
## Step 5: Calculate the Sharpe Ratio

Now we calculate the final scorecard:
$$\text{Sharpe Ratio} = \frac{\text{Average Return}}{\text{Standard Deviation (Risk)}} \times \sqrt{\text{Annualization Factor}}$$

Let's compute this for our strategy!


In [5]:
# Annualization factor (for daily data, 365 days)
ANN_FACTOR = 365

mean_return = strategy_returns.mean()
annual_return = mean_return * ANN_FACTOR

volatility = strategy_returns.std()
annual_volatility = volatility * np.sqrt(ANN_FACTOR)

if annual_volatility > 0:
    sharpe = annual_return / annual_volatility
else:
    sharpe = 0.0

print("=== FINAL STRATEGY SCORECARD ===")
print(f"Average Return per Bar : {mean_return * 100:.2f}%")
print(f"Annualized Return      : {annual_return * 100:.2f}%")
print(f"Annualized Volatility  : {annual_volatility * 100:.2f}%")
print(f"Sharpe Ratio           : {sharpe:.2f}")


=== FINAL STRATEGY SCORECARD ===
Average Return per Bar : -6.88%
Annualized Return      : -2509.57%
Annualized Volatility  : 90.42%
Sharpe Ratio           : -27.75


---
## Practice Challenge For You! (Flip to Reversal)

Remember what we learned: **Reversal is simply $-1.0 \times \text{Momentum}$!**

In the cell below, try running the exact same pipeline, but multiply the signal by `-1.0`:
```python
current_signal = -1.0 * returns.loc[timestamp]
```
What happens to the strategy returns and the Sharpe ratio? Run the cell below to see!


In [6]:
# YOUR TURN: Reversal Strategy (Flipping the sign by -1.0)
rev_weights = pd.DataFrame(np.nan, index=returns.index, columns=returns.columns)

for timestamp in returns.index:
    # FLIP THE SIGN FOR REVERSAL:
    reversal_signal = -1.0 * returns.loc[timestamp]
    
    ranks = reversal_signal.rank()
    centered = ranks - ranks.mean()
    tot_abs = centered.abs().sum()
    if tot_abs > 0:
        rev_weights.loc[timestamp] = centered / tot_abs

# Calculate Reversal Strategy Returns
rev_strategy_returns = (rev_weights.shift(1) * returns).sum(axis=1).dropna()

# Reversal Sharpe Ratio
rev_annual_return = rev_strategy_returns.mean() * ANN_FACTOR
rev_annual_vol = rev_strategy_returns.std() * np.sqrt(ANN_FACTOR)
rev_sharpe = rev_annual_return / rev_annual_vol

print("=== REVERSAL STRATEGY RESULTS ===")
print(f"Momentum Sharpe: {sharpe:.2f}")
print(f"Reversal Sharpe: {rev_sharpe:.2f}")
print("-> Notice the Sharpe ratio is the EXACT OPPOSITE! That's the power of flipping the sign!")


=== REVERSAL STRATEGY RESULTS ===
Momentum Sharpe: -27.75
Reversal Sharpe: 27.75
-> Notice the Sharpe ratio is the EXACT OPPOSITE! That's the power of flipping the sign!
